# Stage 11 — collect the pinned CPU V2 audit report

Attach the output of our Stage 11 CPU audit V2 as notebook input. This read-only collector requires exactly one `stage11_preflight/INPUT_AUDIT.json`, source **362014737**, and audit script **07d5f69e…**. It prints a compact summary and the exact report bytes compressed as gzip/base64 chunks, with SHA256 receipts.

No model weights, upstream/custom code, serializers, training data or test passages are loaded. No GPU is required. Private custom-source export is hash-checked but omitted from logs by default; set `INCLUDE_PRIVATE_CODE_EXPORT=True` only if that separate private text transport is needed. The attached report remains unchanged.


In [ ]:
from pathlib import Path
import base64,gzip,hashlib,json

EXPECTED_AUDIT_SHA256='07d5f69ec0c0002f8679aa1599428b5b2989721e0523cb234040651015898802'
SOURCE_VERSION=362014737
INCLUDE_PRIVATE_CODE_EXPORT=False
CHUNK_CHARACTERS=4000

def sha_bytes(value):
    return hashlib.sha256(value).hexdigest()

def emit_exact_file(label,path,working):
    raw=path.read_bytes()
    compressed=gzip.compress(raw,compresslevel=9,mtime=0)
    encoded=base64.b64encode(compressed).decode('ascii')
    chunks=[encoded[i:i+CHUNK_CHARACTERS] for i in range(0,len(encoded),CHUNK_CHARACTERS)]
    receipt={'path':str(path),'file_bytes':len(raw),'file_sha256':sha_bytes(raw),
             'gzip_bytes':len(compressed),'gzip_sha256':sha_bytes(compressed),
             'base64_characters':len(encoded),'chunks':len(chunks),'chunk_characters':CHUNK_CHARACTERS,
             'transport':'base64(gzip(exact input file bytes,mtime=0))'}
    assert gzip.decompress(base64.b64decode(encoded))==raw
    print(label+'_BEGIN',json.dumps(receipt,separators=(',',':')),flush=True)
    for i,chunk in enumerate(chunks,1):print(label,i,len(chunks),chunk,flush=True)
    print(label+'_END',receipt['file_sha256'],flush=True)
    (working/(label.lower()+'.json.gz')).write_bytes(compressed)
    return receipt

def collect(input_root=Path('/kaggle/input'),working=Path('/kaggle/working')):
    candidates=sorted(p for p in input_root.rglob('INPUT_AUDIT.json') if p.parent.name=='stage11_preflight' and p.is_file())
    if len(candidates)!=1:raise ValueError('Expected exactly one attached Stage11 INPUT_AUDIT.json; found '+str(len(candidates)))
    source=candidates[0]
    if source.stat().st_size>16*1024*1024:raise ValueError('Audit JSON exceeds bounded report collector')
    raw=source.read_bytes();report=json.loads(raw)
    if report.get('stage')!=11 or report.get('source_version')!=SOURCE_VERSION:raise ValueError('Attached report has wrong stage/source version')
    if report.get('audit_script_sha256')!=EXPECTED_AUDIT_SHA256:raise ValueError('Attached report is not pinned CPU V2 audit')
    if report.get('total')!=175 or report.get('mandatory_source_checks_total')!=160:raise ValueError('Attached report has unexpected audit check counts')
    if report.get('status') not in ['PASS_CPU_INPUT_CONTRACT_ONLY','BLOCKED_INPUT_CONTRACT']:raise ValueError('Unknown attached audit status')
    details=report.get('details',{});adapter=details.get('fulltext_final_adapter',{});nf4=details.get('fulltext_nf4_base',{})
    summary={k:report.get(k) for k in ['stage','source_version','status','passed','total','mandatory_source_checks_passed','mandatory_source_checks_total','audit_script_sha256','source_capture_json_sha256','source_manifest_sha256','elapsed_seconds','hashed_file_count','private_code_export_files','private_code_export_sha256','upstream_or_custom_code_executed','torch_transformers_imported','joblib_pickle_torch_deserialized','training_or_inference_started','runtime_feasibility_verified']}
    summary.update(input_audit_file_sha256=sha_bytes(raw),input_audit_file_bytes=len(raw),
                   failed_checks=[c for c in report.get('checks',[]) if c.get('status')!='PASS'],
                   final_adapter={k:adapter.get(k) for k in ['path','bytes','sha256','metadata_sha256','labels','label_alignment','tensors','lora_pairs_matched_to_nf4_metadata','adapter_payload_values_loaded','finite_adapter_values_verified','honest_oof_checkpoint']},
                   nf4_base={k:nf4.get(k) for k in ['path','model_type','hidden_size','layers','source_pinned_files','source_pinned_total_bytes','native_context_tokens']},
                   cpu_environment_and_resources=details.get('cpu_environment_and_resources'),
                   warnings=report.get('warnings',[]))
    private=source.parent/'PRIVATE_CODE_EXPORT.json'
    if private.is_file():
        private_digest=sha_bytes(private.read_bytes())
        if private_digest!=report.get('private_code_export_sha256'):raise ValueError('Attached private code export SHA differs from its audit report')
        summary['private_code_export_actual_sha256']=private_digest
        summary['private_code_export_bytes']=private.stat().st_size
    elif INCLUDE_PRIVATE_CODE_EXPORT:raise ValueError('Requested private code export is missing')
    working.mkdir(parents=True,exist_ok=True)
    print('STAGE11_REPORT_COLLECTION_SUMMARY',json.dumps(summary,separators=(',',':'),allow_nan=False),flush=True)
    receipt={'source_version':SOURCE_VERSION,'expected_audit_script_sha256':EXPECTED_AUDIT_SHA256,
             'summary':summary,'report_transport':emit_exact_file('STAGE11_INPUT_AUDIT_GZIP_B64',source,working),
             'private_code_export_transmitted':INCLUDE_PRIVATE_CODE_EXPORT,
             'model_or_custom_code_imported':False,'input_files_modified':False}
    if INCLUDE_PRIVATE_CODE_EXPORT:receipt['private_code_transport']=emit_exact_file('STAGE11_PRIVATE_CODE_EXPORT_GZIP_B64',private,working)
    (working/'stage11_report_collection.json').write_text(json.dumps(receipt,indent=2,allow_nan=False)+'\n')
    print('STAGE11_REPORT_COLLECTION_FINISHED',report['status'],receipt['report_transport']['file_sha256'],flush=True)
    return receipt

collection_receipt=collect()
